In [1]:
import os
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import torch
import torch.nn as nn
from torch.autograd import Variable
import torch.utils.data as Data
import torchvision
import torch.nn.functional as F
import numpy as np
learning_rate = 1e-4
keep_prob_rate = 0.7 #
max_epoch = 3
BATCH_SIZE = 50

DOWNLOAD_MNIST = False
if not(os.path.exists('./mnist/')) or not os.listdir('./mnist/'):
    # not mnist dir or mnist is empyt dir
    DOWNLOAD_MNIST = True


train_data = torchvision.datasets.MNIST(root='./mnist/',train=True, transform=torchvision.transforms.ToTensor(), download=DOWNLOAD_MNIST,)
train_loader = Data.DataLoader(dataset = train_data ,batch_size= BATCH_SIZE ,shuffle= True)

test_data = torchvision.datasets.MNIST(root = './mnist/',train = False)
test_x = Variable(torch.unsqueeze(test_data.test_data,dim  = 1),volatile = True).type(torch.FloatTensor)[:500]/255.
test_y = test_data.test_labels[:500].numpy()

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d( 
                in_channels= 1,  
                out_channels= 32,
                kernel_size= 7,
                stride= 1,
                padding= 3, # (7-1)/2 = 3
            ),
            nn.ReLU(),        # activation function
            nn.MaxPool2d(2),  # pooling operation(尺寸减半: 28x28->14x14)
        )
        self.conv2 = nn.Sequential( 
            nn.Conv2d(
                in_channels=32, 
                out_channels=64, 
                kernel_size=5, 
                stride=1, 
                padding=2  # (5-1)/2 = 2
            ),
            nn.ReLU(),
            nn.MaxPool2d(2) # (尺寸再次减半: 14x14->7x7)
        )
        
        self.out1 = nn.Linear( 7*7*64 , 1024 , bias= True)   

        self.dropout = nn.Dropout(keep_prob_rate)
        self.out2 = nn.Linear(1024, 10, bias=True)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        
        x = x.view(x.size(0), -1)  
        
        out1 = self.out1(x)
        out1 = F.relu(out1)
        out1 = self.dropout(out1)
        out2 = self.out2(out1)
        
        output = F.softmax(out2, dim=1) 
        return output


def test(cnn):
    global prediction
    y_pre = cnn(test_x)
    _,pre_index= torch.max(y_pre,1)
    pre_index= pre_index.view(-1)
    prediction = pre_index.data.numpy()
    correct  = np.sum(prediction == test_y)
    return correct / 500.0


def train(cnn):
    optimizer = torch.optim.Adam(cnn.parameters(), lr=learning_rate )
    loss_func = nn.CrossEntropyLoss()
    for epoch in range(max_epoch):
        for step, (x_, y_) in enumerate(train_loader):
            x ,y= Variable(x_),Variable(y_)
            output = cnn(x)  
            loss = loss_func(output,y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            if step != 0 and step % 20 ==0:
                print("=" * 10,step,"="*5,"="*5, "test accuracy is ",test(cnn) ,"=" * 10 )

if __name__ == '__main__':
    cnn = CNN()
    train(cnn)




100%|██████████| 9.91M/9.91M [05:21<00:00, 30.8kB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 103kB/s]
100%|██████████| 1.65M/1.65M [00:02<00:00, 817kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.79MB/s]
/home/fanqi/my_ai_env/EvaHan2026/lib/python3.12/site-packages/torchvision/datasets/mnist.py:81: UserWarning: test_data has been renamed data
  warnings.warn("test_data has been renamed data")
/tmp/ipykernel_2394196/110371711.py:27: UserWarning: volatile was removed and now has no effect. Use `with torch.no_grad():` instead.
  test_x = Variable(torch.unsqueeze(test_data.test_data,dim  = 1),volatile = True).type(torch.FloatTensor)[:500]/255.
/home/fanqi/my_ai_env/EvaHan2026/lib/python3.12/site-packages/torchvision/datasets/mnist.py:71: UserWarning: test_labels has been renamed targets
  warnings.warn("test_labels has been renamed targets")


========== 20 ===== ===== test accuracy is  0.21 ==========
========== 40 ===== ===== test accuracy is  0.29 ==========
========== 60 ===== ===== test accuracy is  0.38 ==========
========== 80 ===== ===== test accuracy is  0.506 ==========
========== 100 ===== ===== test accuracy is  0.598 ==========
========== 120 ===== ===== test accuracy is  0.64 ==========
========== 140 ===== ===== test accuracy is  0.702 ==========
========== 160 ===== ===== test accuracy is  0.704 ==========
========== 180 ===== ===== test accuracy is  0.764 ==========
========== 200 ===== ===== test accuracy is  0.79 ==========
========== 220 ===== ===== test accuracy is  0.776 ==========
========== 240 ===== ===== test accuracy is  0.838 ==========
========== 260 ===== ===== test accuracy is  0.844 ==========
========== 280 ===== ===== test accuracy is  0.846 ==========
========== 300 ===== ===== test accuracy is  0.852 ==========
========== 320 ===== ===== test accuracy is  0.872 ==========
========== 340 ==